# Lab 2: Temperature Data Streaming Analysis

## Requirements:
- Read JSON files from streaming directory
- Calculate average temperature per country every 15 minutes
- Discard late data (> 10 minutes late)
- Add files one by one and observe results
- Write output to two locations (console + file system)
- watermark = 10 minutes

In [1]:
print(".")

.


In [2]:
# Cell 1: Spark Session Setup and Directory Cleaning
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, window, avg, to_timestamp
from pyspark.sql.types import StructType, StructField, StringType, DoubleType
import os
import shutil
import time

print("⏳ Step 1: Initializing Spark Session & Cleaning Folders...")

# 1. Create Spark Session with Hive Implementation
spark = SparkSession.builder \
    .appName("TemperatureStreamingLab2") \
    .config("spark.sql.warehouse.dir", "/home/jovyan/lab2/warehouse") \
    .config("spark.sql.catalogImplementation", "hive") \
    .enableHiveSupport() \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")

# 2. Stop any active streams from previous runs
for q in spark.streams.active:
    q.stop()

# 3. Define configuration paths (Including the Warehouse)
input_path = "/home/jovyan/lab2/streaming_input"
source_path = "/home/jovyan/lab2/data"
warehouse_path = "/home/jovyan/lab2/warehouse"
json_output = "/home/jovyan/lab2/output/json"
checkpoint_json = "/home/jovyan/lab2/checkpoints/json"
checkpoint_hive = "/home/jovyan/lab2/checkpoints/hive"
checkpoint_console = "/home/jovyan/lab2/checkpoints/console"

# 4. Automate folder creation and deep cleaning
folders = [input_path, json_output, checkpoint_json, checkpoint_hive, checkpoint_console, warehouse_path]
for folder in folders:
    if os.path.exists(folder):
        for item in os.listdir(folder):
            item_path = os.path.join(folder, item)
            try:
                if os.path.isfile(item_path) or os.path.islink(item_path):
                    os.remove(item_path)
                elif os.path.isdir(item_path):
                    shutil.rmtree(item_path, ignore_errors=True)
            except Exception as e:
                print(f"⚠️ Note: Safe skip on item {item_path}: {e}")
    else:
        # Create directory with exist_ok safety flag
        os.makedirs(folder, exist_ok=True)

print("✓ Environment and Hive Warehouse are clean and ready!")

⏳ Step 1: Initializing Spark Session & Cleaning Folders...
✓ Environment and Hive Warehouse are clean and ready!


In [3]:
# Cell 2: Define Schema and Ingestion Pipeline
print("⏳ Step 2: Defining Schema and Transformations...")

# 1. Define JSON Schema
schema = StructType([
    StructField("event_timestamp", StringType(), True),
    StructField("country", StringType(), True),
    StructField("temperature", DoubleType(), True)
])

# 2. Create Streaming DataFrame from Input Directory
stream_df = spark.readStream \
    .schema(schema) \
    .option("maxFilesPerTrigger", 1) \
    .json(input_path)

# 3. Apply Watermark and 15-Minute Window Grouping
agg_df = stream_df \
    .withColumn("event_time", to_timestamp(col("event_timestamp"), "yyyy-MM-dd HH:mm:ss")) \
    .withWatermark("event_time", "10 minutes") \
    .groupBy(
        window(col("event_time"), "15 minutes"),
        col("country")
    ) \
    .agg(avg("temperature").alias("avg_temperature")) \
    .select(
        col("country"),
        col("window.start").alias("window_start"),
        col("window.end").alias("window_end"),
        col("avg_temperature")
    )

print("✓ Data transformation pipeline is ready!")

⏳ Step 2: Defining Schema and Transformations...
✓ Data transformation pipeline is ready!


In [4]:
# Cell 3: Start Multi-Destination Streaming Queries
print("🚀 Step 3: Starting Streaming Engines to 3 Locations...")

# Location 1: Write to local File System as JSON
json_query = agg_df.writeStream \
    .format("json") \
    .outputMode("append") \
    .option("path", json_output) \
    .option("checkpointLocation", checkpoint_json) \
    .start()

# Location 2: Write to Hive Table (Using Parquet format and toTable mapping)
hive_query = agg_df.writeStream \
    .format("parquet") \
    .outputMode("append") \
    .option("checkpointLocation", checkpoint_hive) \
    .toTable("default.temperature_avg_table")

# Location 3: Print directly to Jupyter Notebook Console for instant monitoring
console_query = agg_df.writeStream \
    .format("console") \
    .outputMode("append") \
    .option("truncate", "false") \
    .option("checkpointLocation", checkpoint_console) \
    .start()

print("✓ All 3 streams are now actively running in the background!")

🚀 Step 3: Starting Streaming Engines to 3 Locations...
✓ All 3 streams are now actively running in the background!


In [5]:
# Cell 4: Automated File Feeding (Batch Simulation)
print("⏳ Step 4: Simulating real-time file arrivals...")

files = ["batch1.json", "batch2.json", "batch3.json", "batch4.json", "batch5.json"]

for file_name in files:
    src_file = os.path.join(source_path, file_name)
    dst_file = os.path.join(input_path, file_name)
    
    if os.path.exists(src_file):
        shutil.copy2(src_file, dst_file)
        print(f"➕ Added to Stream: {file_name}")
        # Wait 12 seconds to let Spark fetch the file and print the update
        time.sleep(12)
    else:
        print(f"❌ Warning: {file_name} not found in source directory.")

print("\n🏁 All batch files have been injected successfully!")

⏳ Step 4: Simulating real-time file arrivals...
➕ Added to Stream: batch1.json
➕ Added to Stream: batch2.json
➕ Added to Stream: batch3.json
➕ Added to Stream: batch4.json
➕ Added to Stream: batch5.json

🏁 All batch files have been injected successfully!


In [6]:
# Cell 5: Stop Streams and Verify Results from Filesystem and Hive
print("⏳ Step 5: Allowing streams to finish processing the last batch...")
# Wait 100 seconds to ensure all streaming data is flushed and checkpoints are updated
time.sleep(100) 

print("⏳ Gathering status of all 3 engines before shutdown...\n")

# Destination 3 Check: Verify if Console engine was alive and active
console_status = "Active ✅" if console_query.isActive else "Stopped ❌"
json_status = "Active ✅" if json_query.isActive else "Stopped ❌"
hive_status = "Active ✅" if hive_query.isActive else "Stopped ❌"

print("🖥️ --- Destination 3: Live Console Sink Status ---")
print(f"Engine Name: {console_query.name if console_query.name else 'Console_Stream'}")
print(f"Status: {console_status}")
print(f"Last Progress Rows/sec: {console_query.lastProgress['inputRowsPerSecond'] if console_query.lastProgress else 0.0}")

print("\n⏳ Stopping all streaming engines smoothly...\n")
# Stop all active streaming queries smoothly
json_query.stop()
hive_query.stop()
console_query.stop()

print("📊 --- Destination 2: Reading from Hive Table ---")
try:
    # Refresh metadata to ensure Hive catalog reflects all newly written files
    spark.sql("REFRESH TABLE default.temperature_avg_table")
    hive_result = spark.sql("SELECT * FROM default.temperature_avg_table ORDER BY country")
    print(f"Total rows in Hive = {hive_result.count()}")
    hive_result.show(100, truncate=False)
except Exception as e:
    print("Failed to read from Hive:", e)

print("\n📊 --- Destination 1: Reading from JSON Directory ---")
if os.path.exists(json_output) and len(os.listdir(json_output)) > 0:
    json_result = spark.read.json(json_output)
    print(f"Total rows in JSON Filesystem = {json_result.count()}")
    json_result.orderBy("country").show(100, truncate=False)
else:
    print("JSON folder is empty or not yet flushed.")

⏳ Step 5: Allowing streams to finish processing the last batch...
⏳ Gathering status of all 3 engines before shutdown...

🖥️ --- Destination 3: Live Console Sink Status ---
Engine Name: Console_Stream
Status: Active ✅
Last Progress Rows/sec: 0.0

⏳ Stopping all streaming engines smoothly...

📊 --- Destination 2: Reading from Hive Table ---
Total rows in Hive = 77
+---------+-------------------+-------------------+------------------+
|country  |window_start       |window_end         |avg_temperature   |
+---------+-------------------+-------------------+------------------+
|Australia|2024-01-15 11:00:00|2024-01-15 11:15:00|32.0              |
|Australia|2024-01-15 10:30:00|2024-01-15 10:45:00|31.2              |
|Australia|2024-01-15 11:30:00|2024-01-15 11:45:00|32.5              |
|Australia|2024-01-15 12:30:00|2024-01-15 12:45:00|33.1              |
|Australia|2024-01-15 11:45:00|2024-01-15 12:00:00|32.7              |
|Australia|2024-01-15 13:30:00|2024-01-15 13:45:00|33.4           